In [1]:
!python --version

Python 3.14.7


# Upload images to Roboflow

After narrowing down the ~14k frames from my Raspberry Pi footage using the yolo model, I want to upload them to Roboflow for annotation and training. I considered the Web UI zip upload or using the CLI but using the Python SDK allowed me to set tags and splits by clip. This reduces the chance of frames of the same cyclist a few seconds apart from ending up in both the training and validation sets.

### General Flow:
 - list clips from footage files
 - decide train and test split
 - loop through images in frames
 - for each frame
     - check valid upload
     - build tags
     - decide train or valid split
     - upload to project with tags and split
 - verify upload successful

In [27]:
import roboflow
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

rf = roboflow.Roboflow(api_key=os.getenv('ROBOFLOW_API_KEY'))
project = rf.workspace().project('bikecounter')

loading Roboflow workspace...
loading Roboflow project...


## Training / Validation Split
I'm using i % 4 to separate clips into training and validation splits to give temporal spread across the clips, every 4th clip lands in the validation set.

In [15]:
candidates_path = Path('candidates')
footage_path = Path('footage')

clips = list(footage_path.glob('*.h264'))

train_clips = []
valid_clips = []

for i, clip in enumerate(sorted(clips)):
    if i % 4 == 0:
        valid_clips.append(clip)
    else:
        train_clips.append(clip)

print(f'{len(train_clips)} clips selected for training')
print('\n'.join([c.stem for c in train_clips]))

print(f'\n{len(valid_clips)} clips selected for validation')
print('\n'.join([c.stem for c in valid_clips]))

12 clips selected for training
clip_20260812_133053
clip_20260812_134554
clip_20260812_141557
clip_20260812_144601
clip_20260812_151604
clip_20260812_153105
clip_20260812_161610
clip_20260812_163112
clip_20260812_164613
clip_20260812_173118
clip_20260812_174620
clip_20260812_181623

4 clips selected for validation
clip_20260812_131551
clip_20260812_143059
clip_20260812_154607
clip_20260812_171617


## Verifying the frame count for each split

In theory we should get 25% of our frames in the validation split but if cyclists are more heavily weighted to rush hour, or coincidentally in our selected clips, we could have an uneven split.

In [16]:
from collections import Counter

counts = Counter(f.stem.rsplit('_', 1)[0] for f in candidates_path.glob('*.jpg'))
train_n = sum(counts[c.stem] for c in train_clips)
valid_n = sum(counts[c.stem] for c in valid_clips)
print(f'train: {train_n}, valid: {valid_n} ({valid_n/(train_n+valid_n):.0%})')

train: 797, valid: 197 (20%)


In [17]:
print(project.type, project.name)

object-detection bikecounter


In [21]:
from datetime import datetime

candidate_frames = sorted(candidates_path.glob('*.jpg'))
valid_stems = {clip.stem for clip in valid_clips}
batch_name = f'mined-{datetime.now():%Y%m%d-%H%M}'

uploaded = []
failed = []

frame = candidate_frames[0]

clip_stem = frame.stem.rsplit('_', 1)[0]
split = 'valid' if clip_stem in valid_stems else 'train'
if project.check_valid_image(str(frame)):
    try:
        project.upload_image(
            image_path = str(frame),
            split=split,
            batch_name=batch_name,
            tag_names=[
                clip_stem,
                'yolo26m',
                'conf-0.05'
            ]
        )
        uploaded.append(str(frame))
        print(f'Uploaded {frame.stem} successfully')
    except Exception as e:
        failed.append((str(frame), str(e)))
        print(f'Failed to upload {frame.stem}: {str(e)}')
else:
    print(f'Image {frame} is invalid, skipping upload')

Uploaded clip_20260812_131551_0047 successfully


In [26]:
from tqdm import tqdm

uploaded = set(uploaded)
failed = set()

for frame in tqdm(candidate_frames):
    clip_stem = frame.stem.rsplit('_', 1)[0]
    split = 'valid' if clip_stem in valid_stems else 'train'
    if str(frame) in uploaded:
        continue
    if project.check_valid_image(str(frame)):
        try:
            project.upload_image(
                image_path = str(frame),
                split=split,
                batch_name=batch_name,
                tag_names=[
                    clip_stem,
                    'yolo26m',
                    'conf-0.05'
                ]
            )
            uploaded.add(str(frame))
            # print(f'Uploaded {frame.stem} successfully')
        except Exception as e:
            failed.add((str(frame), str(e)))
            # print(f'Failed to upload {frame.stem}: {str(e)}')
    else:
        print(f'Image {frame} is invalid, skipping upload')

print(f'Upload complete, {len(uploaded)} images uploaded successfully')
print(f'{len(failed)} images failed')

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 994/994 [00:00<00:00, 493039.05it/s]

Upload complete, 994 images uploaded successfully
0 images failed


In [25]:
for image, error in failed:
    print(f'Image {image} failed: {error}')

Image candidates/clip_20260812_163112_0162.jpg failed: <Response [500]>


## Annotation Decisions

With 994 images successfully uploaded to Roboflow, I now have some decisions to make. Primarily, what counts as a bicycle? Obviously regular bicycles count, as well as electric bikes and cargo bikes. I will not be labeling scooters, while they're important for micro-mobility they are not bikes. I will also not label mopeds or motorcycles, including those that pretend to be e-bikes but are really e-motorcycles.

For edge cases like bicycles being walked, bikes on the bus racks or car racks, I will label them as well. It's important for model training to label everything in frame that's a bicycle as a bicycle. The counting logic later can determine if bikes being walked or on the bus count for my totals.

Bikes partially blocked by the tree, cars, etc. will also be labeled. Bikes partially in frame will also be labeled. Also worth noting that labels should cover bicycles only, not bicycle plus rider. Things like fenders, frame bags, panniers, milk crates will be included, but backpacks, riders clothes, etc. will not.

For negatives, like scooters, I'm deciding not to label them as a second class for now. I only saw 2-3 in my review of the 1,000 frame random sample. The miner was looking for bikes, not scooters, but trying to train a second class from a few examples will likely hurt more than help. If I'm seeing lots of false positives on scooters later, I can re-mine and annotate for scooters.